# 01 — Exploratory Data Analysis
## NASA JPL Asteroid Dataset

This notebook profiles the dataset, examines distributions, missing data patterns, and class separability.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_columns', 50)
%matplotlib inline

## 1. Load and Profile Data

In [ ]:
df = pd.read_csv('../data/raw/dataset.csv', low_memory=False)
print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
df.dtypes

In [ ]:
# Column-level statistics
profile = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2),
    'unique': df.nunique(),
})

numeric_stats = df.describe().T[['mean', 'std', 'min', 'max']]
profile = profile.join(numeric_stats)
profile.sort_values('missing_pct', ascending=False)

## 2. Missing Data Analysis

In [ ]:
from src.utils.visualization import plot_missing_heatmap

fig = plot_missing_heatmap(df)
if fig:
    plt.show()

In [ ]:
# Co-missingness of diameter and albedo
both_missing = df['diameter'].isna() & df['albedo'].isna()
both_present = df['diameter'].notna() & df['albedo'].notna()

print(f'Both diameter & albedo missing: {both_missing.sum():,}')
print(f'Both diameter & albedo present: {both_present.sum():,}')
print(f'Only one missing:              {(~both_missing & ~both_present).sum():,}')

## 3. Target Variable Distributions

In [ ]:
from src.utils.visualization import plot_class_distribution

fig = plot_class_distribution(df['class'])
plt.show()

print('\nClass counts:')
print(df['class'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['H'].dropna().hist(bins=100, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'H (Absolute Magnitude) — {df["H"].notna().sum():,} values')
axes[0].set_xlabel('H')

df['diameter'].dropna().hist(bins=100, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title(f'Diameter — {df["diameter"].notna().sum():,} values')
axes[1].set_xlabel('Diameter (km)')

plt.tight_layout()
plt.show()

In [ ]:
# Log-diameter distribution (this is what the model will predict)
fig, ax = plt.subplots(figsize=(10, 5))
np.log1p(df['diameter'].dropna()).hist(bins=100, ax=ax, color='coral', edgecolor='white')
ax.set_title('log(1 + diameter) Distribution')
ax.set_xlabel('log(1 + diameter)')
plt.tight_layout()
plt.show()

In [ ]:
# H vs diameter relationship
subset = df[df['diameter'].notna() & df['H'].notna()].sample(min(10000, df['diameter'].notna().sum()), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(subset['H'], np.log10(subset['diameter']), c=subset['albedo'], 
                cmap='viridis', alpha=0.3, s=5)
plt.colorbar(sc, label='Albedo')
ax.set_xlabel('H (Absolute Magnitude)')
ax.set_ylabel('log10(Diameter) [km]')
ax.set_title('H vs Diameter (colored by Albedo)')
plt.tight_layout()
plt.show()

## 4. Orbital Element Distributions

In [ ]:
orbital_features = ['e', 'a', 'i', 'om', 'w', 'ma', 'q', 'ad', 'per_y']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, col in zip(axes.flat, orbital_features):
    df[col].dropna().hist(bins=100, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Orbital Element Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
numeric_cols = df[orbital_features + ['H', 'diameter', 'moid', 'rms']].select_dtypes(include=[np.number])

corr = numeric_cols.corr()
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix of Key Features')
plt.tight_layout()
plt.show()

## 6. Class-Conditional Feature Distributions

In [ ]:
# Focus on the main classes
main_classes = df['class'].value_counts().head(8).index.tolist()
df_main = df[df['class'].isin(main_classes)]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, col in zip(axes, ['a', 'e', 'i']):
    for cls in main_classes:
        subset = df_main[df_main['class'] == cls][col]
        subset.hist(bins=80, ax=ax, alpha=0.5, label=cls, density=True)
    ax.set_title(f'{col} by Class')
    ax.legend(fontsize=8)
    ax.set_xlabel(col)
plt.suptitle('Key Orbital Elements by Asteroid Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: semi-major axis vs eccentricity, colored by class
sample = df_main.sample(min(30000, len(df_main)), random_state=42)

fig, ax = plt.subplots(figsize=(12, 8))
for cls in main_classes:
    sub = sample[sample['class'] == cls]
    ax.scatter(sub['a'], sub['e'], alpha=0.3, s=3, label=cls)
ax.set_xlabel('Semi-major axis (AU)')
ax.set_ylabel('Eccentricity')
ax.set_title('Orbital Parameter Space by Class')
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

## 7. Tisserand Parameter

In [ ]:
from src.data.preprocessing import compute_tisserand

df['tisserand_j'] = compute_tisserand(df)

fig, ax = plt.subplots(figsize=(12, 6))
for cls in main_classes:
    sub = df[df['class'] == cls]['tisserand_j']
    sub.hist(bins=80, ax=ax, alpha=0.5, label=cls, density=True)
ax.set_xlabel('Tisserand Parameter (T_J)')
ax.set_title('Tisserand Parameter Distribution by Class')
ax.legend(fontsize=8)
ax.axvline(x=3.0, color='red', linestyle='--', alpha=0.7, label='T_J = 3 (comet/asteroid boundary)')
plt.tight_layout()
plt.show()

## 8. Summary Statistics for Diploma

In [ ]:
print('=== Dataset Summary ===')
print(f'Total asteroids: {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'\n=== Class Distribution ===')
for cls, count in df['class'].value_counts().items():
    print(f'  {cls:>4s}: {count:>8,} ({count/len(df)*100:5.2f}%)')

print(f'\n=== Key Missing Data ===')
for col in ['H', 'diameter', 'albedo', 'pha', 'moid']:
    n_miss = df[col].isna().sum()
    print(f'  {col:>10s}: {n_miss:>8,} missing ({n_miss/len(df)*100:5.2f}%)')

print(f'\n=== Target Statistics ===')
for col in ['H', 'diameter']:
    s = df[col].dropna()
    print(f'  {col}: mean={s.mean():.2f}, std={s.std():.2f}, min={s.min():.2f}, max={s.max():.2f}')